In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
response = model.invoke("Whu do parrots talk?")
response

AIMessage(content='Parrots aren’t really “talking” in the way humans do, but they are exceptionally good at **mimicking** the sounds they hear around them. A few key factors explain why many parrots seem to have a talent for speech:\n\n| Factor | How it helps parrots “talk” |\n|--------|-----------------------------|\n| **Social nature** | In the wild parrots live in large, noisy flocks. They use a wide variety of calls to keep in touch, establish bonds, and signal danger. Mimicking sounds is a natural extension of this social communication. |\n| **Vocal anatomy** | Parrots have a highly flexible syrinx (the bird equivalent of a voice box) and a well‑developed brain region for sound processing. This lets them reproduce a broad range of frequencies, from high‑pitched chirps to human speech sounds. |\n| **Learning ability** | Many parrots are **neurocognitively advanced**—they can solve puzzles, use tools, and understand cause‑and‑effect. Their capacity for **imitation** is a by‑product 

In [6]:
from langchain.tools import tool

@tool

def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"Its sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [7]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool:{tool_call['name']}")
    print(f"Args:{tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks weather in Boston. Need to call function get_weather.', 'tool_calls': [{'id': 'fc_d82faf65-e027-4970-a8c2-c72a37b56e37', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 127, 'total_tokens': 168, 'completion_time': 0.086601557, 'completion_tokens_details': {'reasoning_tokens': 14}, 'prompt_time': 0.027939789, 'prompt_tokens_details': None, 'queue_time': 0.055327215, 'total_time': 0.114541346}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_b9a23305b9', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a18586-7a9c-76ca-a22c-e88e017f2931-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_d82faf65-e027-4970-a8c2-c72a37b56e37', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_

Tool Execution Loops

In [11]:
messages = [{"role":"user","content":"What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
for content in messages:
    print(content)
print(final_response.text)

{'role': 'user', 'content': "What's the weather in Boston?"}
content='' additional_kwargs={'reasoning_content': 'User asks for weather in Boston. Use function get_weather.', 'tool_calls': [{'id': 'fc_91d8d290-3e77-498e-bc4b-7faa1011f4bf', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 126, 'total_tokens': 166, 'completion_time': 0.083997421, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.00470469, 'prompt_tokens_details': None, 'queue_time': 0.211931289, 'total_time': 0.088702111}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_a655021f7d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a1858f-8ef0-72a0-847b-ff8ba65c94ef-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_91d8d290-3e77-498e-bc4b-7faa1011f4bf', 'type': 'too